In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder as SklearnOneHotEncoder, LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


def seed_all(seed=20):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


seed_all(20)



In [2]:
train_path = "/kaggle/input/osic-pulmonary-fibrosis-progression/train.csv"
test_path = "/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv"
sample_sub_path = (
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)




In [3]:
class OneHotEncoder(SklearnOneHotEncoder):
    """Thin wrapper to keep the original interface used in the notebook."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._fitted = False

    def fit(self, X, y=None):
        super().fit(X, y)
        self._fitted = True
        return self

    def transform(self, X, categories=None, index=None, name=""):
        if not self._fitted:
            raise ValueError("Encoder has not been fitted.")
        transformed = super().transform(X)
        # Handle both sparse matrix and dense ndarray outputs
        if hasattr(transformed, "toarray"):
            arr = transformed.toarray()
        else:
            arr = transformed
        cols = [f"{name}_{cat}" for cat in categories]
        return pd.DataFrame(arr, columns=cols, index=index)

    def fit_transform(self, X, categories, index, name):
        self.fit(X)
        return self.transform(X, categories=categories, index=index, name=name)


class DataPreparation:
    def __init__(self):
        self.le_sex = LabelEncoder()
        self.le_smoke = LabelEncoder()
        # Use default sparse output (True) so transform works with .toarray()
        self.ohe_smoke = OneHotEncoder(handle_unknown="ignore")
        self.fitted = False

    def __call__(self, df):
        df = df.copy()
        if not self.fitted:
            df["Sex"] = self.le_sex.fit_transform(df["Sex"])
        else:
            df["Sex"] = self.le_sex.transform(df["Sex"])
        if not self.fitted:
            df["SmokingStatus"] = self.le_smoke.fit_transform(df["SmokingStatus"])
        else:
            df["SmokingStatus"] = self.le_smoke.transform(df["SmokingStatus"])
        smoke_cat = self.le_smoke.classes_
        if not self.fitted:
            ohe_df = self.ohe_smoke.fit_transform(
                df[["SmokingStatus"]],
                categories=range(len(smoke_cat)),
                index=df.index,
                name="SmokingStatus",
            )
            self.fitted = True
        else:
            ohe_df = self.ohe_smoke.transform(
                df[["SmokingStatus"]],
                categories=range(len(smoke_cat)),
                index=df.index,
                name="SmokingStatus",
            )
        df = pd.concat([df.drop(columns=["SmokingStatus"]), ohe_df.astype(int)], axis=1)
        return df


prep = DataPreparation()



In [4]:
# Preprocess training data
train_processed = prep(train_df)

FEATURE_COLS = ["Weeks", "Age", "Sex"] + [
    c for c in train_processed.columns if c.startswith("SmokingStatus_")
]

X = train_processed[FEATURE_COLS]
y = train_processed["FVC"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=train_processed["Patient"].astype("category").cat.codes,
)



In [5]:
model = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42
)

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
print("Validation MAE:", mean_absolute_error(y_val, val_pred))



Validation MAE: 428.8523556567727


In [6]:
# Prepare test data for prediction
sample_sub["Patient"] = sample_sub["Patient_Week"].str.extract(r"(.*)_.*")
sample_sub["Weeks"] = sample_sub["Patient_Week"].str.extract(r".*_(.*)").astype(int)

test_merged = sample_sub.merge(
    test_df, on="Patient", how="left", suffixes=("", "_base")
)

test_processed = prep(test_merged)

X_test = test_processed[FEATURE_COLS]

pred_fvc = model.predict(X_test)

CONST_CONFIDENCE = 100.0
conf = np.full_like(pred_fvc, CONST_CONFIDENCE, dtype=float)

submission = pd.DataFrame(
    {"Patient_Week": sample_sub["Patient_Week"], "FVC": pred_fvc, "Confidence": conf}
)

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
